<a href="https://colab.research.google.com/github/bsrikanth24/Best-websites-a-programmer-should-visit/blob/master/rowsBetween_vs_rangeBetween.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.appName("CreateDataFrames").getOrCreate()

# -------------------------
# df1
# -------------------------
data1 = [
    ("A", 1, 1),
    ("B", 2, 2)
]

columns1 = ["ID", "A", "B"]

df1 = spark.createDataFrame(data1, columns1)

# -------------------------
# df2
# -------------------------
data2 = [
    ("A", 3, 333),
    ("B", 4, 444)
]

columns2 = ["ID", "C", "D"]

df2 = spark.createDataFrame(data2, columns2)

# -------------------------
# df3
# -------------------------
data3 = [
    ("A", 555, 5),
    ("B", 666, 6)
]

columns3 = ["ID", "E", "F"]

df3 = spark.createDataFrame(data3, columns3)

# Show DataFrames
df1.show()
df2.show()
df3.show()

+---+---+---+
| ID|  A|  B|
+---+---+---+
|  A|  1|  1|
|  B|  2|  2|
+---+---+---+

+---+---+---+
| ID|  C|  D|
+---+---+---+
|  A|  3|333|
|  B|  4|444|
+---+---+---+

+---+---+---+
| ID|  E|  F|
+---+---+---+
|  A|555|  5|
|  B|666|  6|
+---+---+---+



In [ ]:
df = df1.join(df2, 'ID').join(df3, 'ID')
df.show()
# df = df1.join(df2, on="ID", how="left")
# df.show()  Not working

+---+---+---+---+---+---+---+
| ID|  A|  B|  C|  D|  E|  F|
+---+---+---+---+---+---+---+
|  A|  1|  1|  3|333|555|  5|
|  B|  2|  2|  4|444|666|  6|
+---+---+---+---+---+---+---+



In [ ]:
df = spark.createDataFrame( [(1,2,"a"),(3,2,"a"),(1,3,"b"),(2,2,"a"),(2,3,"b")],
                                 ["time", "value", "class"] )
df.show()

from pyspark.sql import Window
from pyspark.sql import functions as F

windowval = (Window.partitionBy('class').orderBy('time')
            #  .rowsBetween(Window.unboundedPreceding, 0))
             .rowsBetween(Window.unboundedPreceding, Window.currentRow))
df_w_cumsum = df.withColumn('cum_sum', F.sum('value').over(windowval))
df_w_cumsum.show()

+----+-----+-----+
|time|value|class|
+----+-----+-----+
|   1|    2|    a|
|   3|    2|    a|
|   1|    3|    b|
|   2|    2|    a|
|   2|    3|    b|
+----+-----+-----+

+----+-----+-----+-------+
|time|value|class|cum_sum|
+----+-----+-----+-------+
|   1|    2|    a|      2|
|   2|    2|    a|      4|
|   3|    2|    a|      6|
|   1|    3|    b|      3|
|   2|    3|    b|      6|
+----+-----+-----+-------+



In [ ]:
df1 = spark.createDataFrame( [(1,2,"a"),(3,2,"a"),(1,3,"b"),(2,2,"a"),(2,3,"b")],
                                 ["time", "value", "class"] )
df1.show()

from pyspark.sql import Window
from pyspark.sql import functions as F

windowval1 = (Window.partitionBy('class').orderBy('time')
             .rangeBetween(Window.unboundedPreceding, 0))
df_w_cumsum = df1.withColumn('cum_sum', F.sum('value').over(windowval1))
df_w_cumsum.show()

+----+-----+-----+
|time|value|class|
+----+-----+-----+
|   1|    2|    a|
|   3|    2|    a|
|   1|    3|    b|
|   2|    2|    a|
|   2|    3|    b|
+----+-----+-----+

+----+-----+-----+-------+
|time|value|class|cum_sum|
+----+-----+-----+-------+
|   1|    2|    a|      2|
|   2|    2|    a|      4|
|   3|    2|    a|      6|
|   1|    3|    b|      3|
|   2|    3|    b|      6|
+----+-----+-----+-------+



In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# Notice the two rows with time=1 and class="a"
df_ties = spark.createDataFrame(
    [(1, 10, "a"), (1, 20, "a"), (2, 30, "a")],
    ["time", "value", "class"]
)

df_ties.show()

# Create a window specification

window_rows = (Window.partitionBy('class')
               .orderBy('time')
               .rowsBetween(Window.unboundedPreceding, 0))

window_range = (Window.partitionBy('class')
                .orderBy('time')
                .rangeBetween(Window.unboundedPreceding, 0))

print("--- rowsBetween ---")
df_ties.withColumn('cum_sum', F.sum('value').over(window_rows)).show()

print("--- rangeBetween ---")
df_ties.withColumn('cum_sum', F.sum('value').over(window_range)).show()

+----+-----+-----+
|time|value|class|
+----+-----+-----+
|   1|   10|    a|
|   1|   20|    a|
|   2|   30|    a|
+----+-----+-----+

--- rowsBetween ---
+----+-----+-----+-------+
|time|value|class|cum_sum|
+----+-----+-----+-------+
|   1|   10|    a|     10|
|   1|   20|    a|     30|
|   2|   30|    a|     60|
+----+-----+-----+-------+

--- rangeBetween ---
+----+-----+-----+-------+
|time|value|class|cum_sum|
+----+-----+-----+-------+
|   1|   10|    a|     30|
|   1|   20|    a|     30|
|   2|   30|    a|     60|
+----+-----+-----+-------+



In [ ]:
# Source - https://stackoverflow.com/a/66914820
# Posted by Nikunj Kakadiya
# Retrieved 2026-07-24, License - CC BY-SA 4.0

from pyspark.sql.functions import *
from pyspark.sql.types import *
data1 = spark.read.text("/content/datawprld.json5")
schema = StructType(
    [
        StructField('id', StringType(), True),
        StructField('name', StringType(), True),
        StructField('hometown',StringType(),True)
    ]
)
# data2 = data1.withColumn("JsonKey",split(col("value"),"\\[")[0]).withColumn("JsonValue",split(col("value"),"\\[")[1]).withColumn("data",from_json("JsonKey",schema)).select(col('data.*'),'JsonValue')
data2 = data1.withColumn("JsonKey",split(col("value"),"\[")[0]).withColumn("JsonValue",regexp_replace(split(col("value"),"\[")[1],"]","")).withColumn("data",from_json("JsonKey",schema)).select(col('data.*'),'JsonValue')


<>:16: SyntaxWarning: invalid escape sequence '\['
<>:16: SyntaxWarning: invalid escape sequence '\['
<>:16: SyntaxWarning: invalid escape sequence '\['
<>:16: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_896/3057060311.py:16: SyntaxWarning: invalid escape sequence '\['
  data2 = data1.withColumn("JsonKey",split(col("value"),"\[")[0]).withColumn("JsonValue",regexp_replace(split(col("value"),"\[")[1],"]","")).withColumn("data",from_json("JsonKey",schema)).select(col('data.*'),'JsonValue')
/tmp/ipykernel_896/3057060311.py:16: SyntaxWarning: invalid escape sequence '\['
  data2 = data1.withColumn("JsonKey",split(col("value"),"\[")[0]).withColumn("JsonValue",regexp_replace(split(col("value"),"\[")[1],"]","")).withColumn("data",from_json("JsonKey",schema)).select(col('data.*'),'JsonValue')


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


spark = SparkSession.builder.getOrCreate()
df = spark.read.json("/content/dataworld", multiLine=True)
df = df.withColumn("tmp", F.explode_outer("itemList"))
df = df.select(["customer", "tmp.item", "tmp.price", "tmp.quantity"])
df.show(10, False)
df.printSchema()

+--------+--------------+-----+--------+
|customer|item          |price|quantity|
+--------+--------------+-----+--------+
|10      |20126907_EA   |1.88 |1.0     |
|10      |20185742_EA   |0.99 |1.0     |
|10      |20138681_EA   |1.79 |1.0     |
|10      |20049778001_EA|2.47 |1.0     |
|10      |20419715007_EA|3.33 |1.0     |
|10      |20321434_EA   |2.47 |1.0     |
|10      |20068076_KG   |28.24|10.086  |
|10      |20022893002_EA|1.77 |1.0     |
|10      |20299328003_EA|1.25 |1.0     |
+--------+--------------+-----+--------+

root
 |-- customer: long (nullable = true)
 |-- item: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: double (nullable = true)



In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


spark = SparkSession.builder.getOrCreate()
df = spark.read.json("/content/dataworld", multiLine=True)
df = df.withColumn("tmp", F.explode_outer("itemList"))
df = df.select(["customer", "tmp.item", "tmp.price"])
df.show(10, False)
df.printSchema()


+--------+--------------+-----+
|customer|item          |price|
+--------+--------------+-----+
|10      |20126907_EA   |1.88 |
|10      |20185742_EA   |0.99 |
|10      |20138681_EA   |1.79 |
|10      |20049778001_EA|2.47 |
|10      |20419715007_EA|3.33 |
|10      |20321434_EA   |2.47 |
|10      |20068076_KG   |28.24|
|10      |20022893002_EA|1.77 |
|10      |20299328003_EA|1.25 |
+--------+--------------+-----+

root
 |-- customer: long (nullable = true)
 |-- item: string (nullable = true)
 |-- price: double (nullable = true)

